# 📊 Data Jobs Market Tracker — Dashboard

Global jobs related to Data Analysis (Data Analyst, Data Engineer, Data Scientist, BI, ML Engineer) through Adzuna API in 7 countries: US, GB, DE, BR, ES, NL, IT.

**Pipeline:** Adzuna API → pandas cleaning → PostgreSQL → this dashboard.

| Stage | File |
|---|---|
| Extraction | `adzuna_api.py` |
| Cleaning | `clean.py` |
| Loading  to a DB | `load_to_db.py` |
| SQL Analysis | `analysis.sql` |
| Visualization | `dashboard.ipynb` |


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import HTML, display
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

# IMPORTANT: nbconvert cannot represent the native Plotly mimetype
# (application/vnd.plotly.v1+json) produced by fig.show(). Instead of
# relying on a Plotly renderer, we render each figure to static HTML/JS
# ourselves and display it as text/html, which nbconvert always supports.
_plotlyjs_included = {"done": False}

def show_fig(fig):
    """Display a Plotly figure as embedded HTML (works in Jupyter and in nbconvert exports)."""
    include_js = "cdn" if not _plotlyjs_included["done"] else False
    _plotlyjs_included["done"] = True
    display(HTML(fig.to_html(include_plotlyjs=include_js, full_html=False)))

load_dotenv()

DB_USER     = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "postgres")
DB_HOST     = os.getenv("DB_HOST", "localhost")
DB_PORT     = os.getenv("DB_PORT", "5432")
DB_NAME     = os.getenv("DB_NAME", "adventureworks")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

df = pd.read_sql("SELECT * FROM dw.data_jobs_market", engine)
print(f"Registros cargados: {len(df):,}")
df.head()

Registros cargados: 4,389


,job_id,title,seniority,company,location,country,is_remote,salary_min,salary_max,salary_avg,category,contract_type,published_date,search_term,description,url,scraped_at,loaded_at
0,5767745419,Data Analyst,Not specified,Trane Technologies,"Noblesville, Hamilton County",US,False,147616.97,147616.97,147616.97,IT Jobs,None,2026-06-18,data analyst,Be a part of our mission! As a world leader in...,https://www.adzuna.com/land/ad/5767745419?se=9...,2026-06-18T23:20:06.652649,2026-06-18T23:44:58.650353
1,5762463670,Wargame Data Analyst,Not specified,ManTech International,"Triangle, Prince William County",US,False,125000.00,165000.00,145000.00,IT Jobs,None,2026-06-13,data analyst,As a Wargame Data Analyst in ManTech Internati...,https://www.adzuna.com/land/ad/5762463670?se=9...,2026-06-18T23:20:06.652649,2026-06-18T23:44:58.650353
2,5768374476,Sr Data Analyst,Senior,The Job Connections Project,"Westwood, Norfolk County",US,False,74333.16,74333.16,74333.16,Admin Jobs,None,2026-06-18,data analyst,We are seeking a Senior Data Analyst for our W...,https://www.adzuna.com/land/ad/5768374476?se=9...,2026-06-18T23:20:06.652649,2026-06-18T23:44:58.650353
3,5749302042,Data Analyst,Not specified,ENNOVARA,"Las Vegas, Clark County",US,False,66047.93,66047.93,66047.93,IT Jobs,None,2026-06-02,data analyst,Job Description Job Description Position Title...,https://www.adzuna.com/land/ad/5749302042?se=9...,2026-06-18T23:20:06.652649,2026-06-18T23:44:58.650353
4,5768256011,Data Analyst,Not specified,Trenton Health Team Inc,"Trenton, Mercer County",US,False,124535.55,124535.55,124535.55,IT Jobs,None,2026-06-18,data analyst,Job Description Job Description Are you ready ...,https://www.adzuna.com/land/ad/5768256011?se=9...,2026-06-18T23:20:06.652649,2026-06-18T23:44:58.650353


## 1. Jobs Volume by Countries

In [8]:
country_summary = (
    df.groupby("country")
      .agg(total_jobs=("job_id", "count"),
           remote_jobs=("is_remote", "sum"),
           with_salary=("salary_avg", lambda x: x.notna().sum()))
      .sort_values("total_jobs", ascending=False)
      .reset_index()
)

fig = px.bar(
    country_summary,
    x="country", y="total_jobs",
    color="total_jobs",
    color_continuous_scale="Blues",
    text="total_jobs",
    title="Jobs Postings by Country",
    labels={"country": "País", "total_jobs": "Total Job Postings"}
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, height=450)
show_fig(fig)

## 2. Most Demanding Roles

In [ ]:
role_demand = (
    df.groupby("search_term")
      .size()
      .sort_values(ascending=False)
      .reset_index(name="postings")
)

fig = px.bar(
    role_demand,
    x="postings", y="search_term",
    orientation="h",
    color="postings",
    color_continuous_scale="Teal",
    title="Total Most in-Demand Jobs",
    labels={"search_term": "Role", "postings": "Job Postings"}
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=450, showlegend=False)
show_fig(fig)

## 3. Salaries by seniority

In [4]:
salary_df = df[
    (df["salary_avg"].notna()) &
    (df["salary_avg"] > 10000) &
    (df["seniority"] != "Not specified")
]

fig = px.box(
    salary_df,
    x="seniority", y="salary_avg",
    color="seniority",
    category_orders={"seniority": ["Junior", "Mid", "Senior"]},
    title="Salary Distribution by Seniority (USD)",
    labels={"seniority": "Seniority", "salary_avg": "Average Salary (USD)"}
)
fig.update_layout(height=500, showlegend=False)
show_fig(fig)

## 4. Average Salary by Role

In [ ]:
role_salary = (
    salary_df.groupby("search_term")["salary_avg"]
    .agg(["mean", "median", "count"])
    .query("count >= 5")
    .sort_values("median", ascending=False)
    .reset_index()
)

fig = px.bar(
    role_salary,
    x="median", y="search_term",
    orientation="h",
    color="median",
    color_continuous_scale="Sunset",
    text=role_salary["median"].round(0).astype(int).astype(str),
    title="Average Salary by Role",
    labels={"search_term": "Role", "median": "Average Salary (USD)"}
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=450, showlegend=False)
show_fig(fig)

## 5. Seniority by Country

In [6]:
seniority_country = (
    df[df["seniority"] != "Not specified"]
    .groupby(["country", "seniority"])
    .size()
    .reset_index(name="count")
)

fig = px.bar(
    seniority_country,
    x="country", y="count", color="seniority",
    barmode="stack",
    category_orders={"seniority": ["Junior", "Mid", "Senior"]},
    color_discrete_map={"Junior": "#90CAF9", "Mid": "#42A5F5", "Senior": "#1565C0"},
    title="Seniority by country",
    labels={"country": "Country", "count": "Jobs Postings Amount", "seniority": "Level"}
)
fig.update_layout(height=450)
show_fig(fig)

## 6. % remote Job Postings by role 

In [ ]:
remote_by_role = (
    df.groupby("search_term")
      .agg(total=("job_id", "count"), remote=("is_remote", "sum"))
      .assign(remote_pct=lambda x: (x["remote"] / x["total"] * 100).round(1))
      .sort_values("remote_pct", ascending=False)
      .reset_index()
)

fig = px.bar(
    remote_by_role,
    x="remote_pct", y="search_term",
    orientation="h",
    color="remote_pct",
    color_continuous_scale="Greens",
    text=remote_by_role["remote_pct"].astype(str) + "%",
    title="% of remote job postings by role",
    labels={"search_term": "Role", "remote_pct": "% remote"}
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=450, showlegend=False)
show_fig(fig)

---

### Key insights

- **US and UK** concentrate the highest volume of job openings and are the markets with the best salary reporting.
- The salary jump **Junior → Senior** is approximately 2x in the median.
- **Data Engineer** is the role with the highest absolute demand; **Data Scientist** and **ETL Developer** offer the highest median salaries.
- **Germany (DE)** has a high volume of job openings but low salary transparency (few listings publish salaries).
- The Netherlands (NL) shows the lowest proportion of remote work, suggesting a more in-office work culture.

---
*Proyecto desarrollado por Juan Bautista Acuña — [GitHub](https://github.com/bautistaacuna/DataAnalysis) | [Tableau Public](https://public.tableau.com/app/profile/juan.bautista.acuna)*
